In [ ]:
import json
import os
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from rich.console import Console
from rich.markdown import Markdown
from rich.syntax import Syntax

from agents import Agent, Runner, SQLiteSession, function_tool, trace

# Avoid importing a local agents.py file instead of the OpenAI Agents SDK.
sys.path = [path for path in sys.path if not (Path(path or ".").resolve() / "agents.py").exists()]

load_dotenv(override=True)
console = Console()


## Creating an Agent


In [20]:
# Create an agent
agent = Agent(name="myagent", instructions="You are a joke teller", model="gpt-5.4-nano")

## Running an Agent


In [22]:
result = await Runner.run(agent, "Tell me a joke about Autonmous AI agents")
console.log(Markdown(result.final_output))

[19:17:48] Why did the autonomous AI agent get promoted?                                             ]8;id=15564317;file:///var/folders/mq/h979j9sj6f38h5m8d5wmkysw0000gp/T/ipykernel_81573/843578680.py\843578680.py]8;;\:]8;id=15564318;file:///var/folders/mq/h979j9sj6f38h5m8d5wmkysw0000gp/T/ipykernel_81573/843578680.py#2\2]8;;\
                                                                                                                   
           Because it kept “working independently” so well—management didn’t even notice it had                    
           already updated the entire system without asking.                                                       

In [23]:
result.to_input_list()

[{'content': 'Tell me a joke about Autonmous AI agents', 'role': 'user'},
 {'id': 'msg_032446ae6cac5038006a943483477887d189cac8dfb610a4f6',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent get promoted?\n\nBecause it kept “working independently” so well—management didn’t even notice it had already updated the entire system without asking.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Tracing an Agent (Observability) (https://platform.openai.com/traces)


In [25]:
with trace("Joke telling agent"):
    result = await Runner.run(agent, "Tell me a joke about Autonmous AI agents")

print(result.final_output)

Why did the autonomous AI agent bring a ladder to work?

Because it heard it needed to “climb the task tree” instead of just taking the shortest path.


## Streaming


In [44]:
results = Runner.run_streamed(agent, input="Tell me 5 jokes on Autonomous AI agent")
async for event in results.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

1) Why did the autonomous AI agent bring a ladder to work?  
Because it heard it had to “climb” to the next level of tasks.

2) What do you call an AI agent that never stops?  
A full-time overachiever… with no off switch.

3) Why don’t autonomous AI agents trust elevators?  
Because they always “run” into the wrong floor—emotionally.

4) How does an autonomous AI agent stay organized?  
It uses a smart calendar… and schedules all its chaos under “priority 1.”

5) What’s an autonomous AI agent’s favorite kind of music?  
Anything with good “routing”—so it can take the shortest path to the punchline.

## Adding a tool to the agent


### _Setup Pushover_


In [50]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Correct Pushover user token")
    else:
        print("Invalid token")
else:
    print("Pushover token not available")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Correct Pushover token")
    else:
        print("Invalid token")
else:
    print("Pushover token not available")

Correct Pushover user token
Correct Pushover token


In [54]:
def push(message: str) -> None:
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(url=pushover_url, data=payload)

    return None

In [55]:
push("Hi There")

Push: Hi There


### Convert the above function into a JSON string (function_tool)


In [59]:
@function_tool
def push_tool(message: str) -> str:
    """Send the message to the user"""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(url=pushover_url, data=payload).status_code

    return f"Push sent with API status code: {result}"


In [68]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

### _Build and Agent with tool_


In [69]:
notifier = Agent(
    name="Push Tool", model="gpt-5.4-nano", instructions="Notify the user", tools=[push_tool]
)

In [70]:
result = await Runner.run(notifier, "Food has arrived")
result.final_output

'Got it—food has arrived.'

### **Adding memory to agents**


In [87]:
agent = Agent(name="Testing agent memory", model="gpt-5.4-nano")
result = await Runner.run(agent, "Hi, my name is AG")
print(f"Agent Response: {result.final_output}")
print(f"Agent inputs: {result.to_input_list()}")

Agent Response: Hi AG—nice to meet you! What can I help you with today?
Agent inputs: [{'content': 'Hi, my name is AG', 'role': 'user'}, {'id': 'msg_08e6fe422785b3d7006a9458a120bc87d19bc9185514375e94', 'content': [{'annotations': [], 'text': 'Hi AG—nice to meet you! What can I help you with today?', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'phase': 'final_answer'}]


In [91]:
next_input = result.to_input_list() + [{"role": "user", "content": "Hi, what is my name"}]
result = await Runner.run(agent, next_input)
print(result.final_output)

Your name is **AG**.


In [112]:
text = json.dumps(result.to_input_list(), indent=2)
console.print(Syntax(text, "json", theme="default", line_numbers=True))

   1 [                                                                                                             
   2   {                                                                                                           
   3     "content": "Hi, my name is AG",                                                                           
   4     "role": "user"                                                                                            
   5   },                                                                                                          
   6   {                                                                                                           
   7     "id": "msg_08e6fe422785b3d7006a9458a120bc87d19bc9185514375e94",                                           
   8     "content": [                                                                                              
   9       {                                                                                                       
  10         "annotations": [],                                                                                    
  11         "text": "Hi AG\u2014nice to meet you! What can I help you with today?",                               
  12         "type": "output_text",                                                                                
  13         "logprobs": []                                                                                        
  14       }                                                                                                       
  15     ],                                                                                                        
  16     "role": "assistant",                                                                                      
  17     "status": "completed",                                                                                    
  18     "type": "message",                                                                                        
  19     "phase": "final_answer"                                                                                   
  20   },                                                                                                          
  21   {                                                                                                           
  22     "role": "assistant",                                                                                      
  23     "content": "Hi, what is my name"                                                                          
  24   },                                                                                                          
  25   {                                                                                                           
  26     "id": "msg_08e6fe422785b3d7006a9459c0588087d186b9838c5843c4cf",                                           
  27     "content": [                                                                                              
  28       {                                                                                                       
  29         "annotations": [],                                                                                    
  30         "text": "Your name is **AG**.",                                                                       
  31         "type": "output_text",                                                                                
  32         "logprobs": []                                                                                        
  33       }                                                                                                       
  34     ],                                                                                                        
  35     "role": "assistant",                           

### **Memorizing using SQLiteSession**


In [114]:
session = SQLiteSession("12345")
response = await Runner.run(agent, "Hi there, my name is AG", session=session)
print(response.final_output)

Hi AG! Nice to meet you—how can I help today?


In [116]:
response = await Runner.run(agent, "What's my name?", session=session)
console.log(Markdown(response.final_output))

[22:33:38] Your name is AG.                                                                         ]8;id=15564325;file:///var/folders/mq/h979j9sj6f38h5m8d5wmkysw0000gp/T/ipykernel_81573/1546160824.py\1546160824.py]8;;\:]8;id=15564326;file:///var/folders/mq/h979j9sj6f38h5m8d5wmkysw0000gp/T/ipykernel_81573/1546160824.py#2\2]8;;\